<a href="https://colab.research.google.com/github/jingxuchen19/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Class%2019/lab_ch19_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 30 min Core + 15 min Extension + SHAP Deep Dive

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [6]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Part 1: Find the Bug — Model Comparison (10 min)

The following code trains three models and reports their performance.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain.

In [7]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: Model comparison — find the bug
# -----------------------------------------------------------

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# BUG IS HERE: RF is evaluated on TRAINING data, not test data
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_train, rf.predict(X_train)):.4f}")  # \u2190 WRONG: using training set
print()
print('Conclusion: Random Forest achieves R\u00b2 > 0.97! Far superior to alternatives.')

=== Model Comparison ===
Single Tree  — R²: 0.6221
Ridge        — R²: 0.5759
Random Forest — R²: 0.9736

Conclusion: Random Forest achieves R² > 0.97! Far superior to alternatives.


### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and error type)

The RF's R² is computed on training data (y_train, X_train), instead of test data. The line r2_score(y_train, rf.predict(X_train)) should be r2_score(y_test, rf.predict(X_test))

2. **Why is this dangerous?** (what misleading conclusion does it lead to?)

It makes RF look way better than it actually is (R² = 0.97). RF with default settings tends to overfit the training set, so evaluating on training data gives an inflated score. This leads to a misleading conclusion that RF is "far superior" when the real gpa is much smaller.

3. **Fix the code below** and report the correct R²

**Verification checkpoint:** After fixing, the RF Test R² should be between 0.78 and 0.83. If you get >0.95, you haven't found the bug.

After fixing, the corrected Test R² values are: Single Tree R²= 0.6221
Ridge R²= 0.5759, Random Forest R²= 0.8051. RF still performs best, but the gap is much smaller than the inflated 0.97 suggested.

4. **Which chapter concept does this error violate?** (hint: Ch 15)

This violates the train/test separation principle from Ch 15. We should never evaluate model performance on the same data used to fit it. That measures memorization, not generalization.

In [8]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Fix the model comparison bug from Part 1
# -----------------------------------------------------------

# YOUR FIX HERE
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

# fixed, all models evaluated on test data
print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_test, rf.predict(X_test)):.4f}")  # now using test data

=== Model Comparison ===
Single Tree  — R²: 0.6221
Ridge        — R²: 0.5759
Random Forest — R²: 0.8051


## Part 2: Find the Methodological Flaw — Feature Importance (10 min)

The following analysis uses feature importance to make a **causal claim**.
The code runs correctly. The methodology is wrong. Find the flaw.

In [9]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Feature importance with flawed causal reasoning
# -----------------------------------------------------------

rf_correct = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature Importance (MDI):')
print(importance.round(4))
print()
print('POLICY RECOMMENDATION:')
print(f'The top predictor is {importance.index[0]} (importance = {importance.iloc[0]:.3f}).')
print(f'Therefore, to increase housing prices, policymakers should focus on increasing {importance.index[0]}.')
print(f'The second most important lever is {importance.index[1]}.')

Feature Importance (MDI):
MedInc        0.5259
AveOccup      0.1381
Latitude      0.0886
Longitude     0.0883
HouseAge      0.0544
AveRooms      0.0444
Population    0.0307
AveBedrms     0.0296
dtype: float64

POLICY RECOMMENDATION:
The top predictor is MedInc (importance = 0.526).
Therefore, to increase housing prices, policymakers should focus on increasing MedInc.
The second most important lever is AveOccup.


### YOUR DIAGNOSIS

1. **What is the methodological flaw?** (the code is correct — the reasoning is wrong)

The code treats feature importance as causal evidence. MDI only measures how useful a feature is for prediction, not whether changing that feature would cause house prices to go up. Saying "increase MedInc to raise prices" is a causal claim that MDI cannot support.

2. **Why can't we use MDI for policy recommendations?** (connect to Ch 10 DAGs and Ch 15 prediction vs. explanation)

MDI captures correlation/predictive power, not causation. MedInc could be high because wealthy people move to expensive areas (reverse causality), or because of confounders like school quality or location that drive both income and prices. Without controlling for these, we can't make policy recommendations from importance scores.

3. **What would you need to make a causal claim?** (hint: Ch 24 DML)

You'd need a causal identification strategy, something like an instrument, a natural experiment, or Double ML that isolates the causal effect of MedInc on house prices while controlling for confounders.

4. **Bonus:** MDI has a known statistical bias. What is it, and what alternative would you use?

MDI is biased toward high-cardinality and continuous features because they offer more possible split points. A better alternative is permutation importance, which measures how much test performance drops when a feature is randomly shuffled. It's model-agnostic and less biased.

**Verification checkpoint:** Your diagnosis should mention at least: (a) prediction ≠ causation, (b) confounding/omitted variables, (c) MDI bias toward high-cardinality features.

In [10]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Run permutation importance and write a proper (non-causal)
# interpretation of the results
# -----------------------------------------------------------

# YOUR CORRECTED ANALYSIS HERE
# permutation importance as a less biased alternative to MDI
perm_result = permutation_importance(
    rf_correct, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE
)

perm_imp = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation Importance:")
print(perm_imp.round(4))

# non-causal interpretation: MedInc is the strongest predictor of housing prices,
# but this does not mean raising income would cause prices to go up.
# Feature importance only tells us what's useful for prediction,
# not what causes the outcome.

Permutation Importance:
MedInc        0.7347
Latitude      0.4429
Longitude     0.3352
AveOccup      0.2036
HouseAge      0.0721
AveRooms      0.0271
AveBedrms     0.0095
Population    0.0087
dtype: float64


## Part 3: Hyperparameter Tuning + XGBoost Comparison (10 min)

Tune the RF, then compare against XGBoost (gradient boosting).

In [11]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune RF with GridSearchCV and compare with GBR
# -----------------------------------------------------------

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5],
}

# 1. GridSearchCV on RandomForestRegressor
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid, cv=3, scoring='neg_mean_squared_error'
)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

# 2. Fit GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1)
gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=5,
    learning_rate=0.1, random_state=RANDOM_STATE
)
gbr.fit(X_train, y_train)

# 3. Compare Test RMSE and R\u00b2 for: Ridge, RF (default), RF (tuned), GBR
models = {
    'Ridge': ridge,
    'RF (default)': rf,
    'RF (tuned)': best_rf,
    'GBR': gbr
}

print("\n=== Model Comparison ===")
for name, model in models.items():
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name:<15} — RMSE: {rmse:.4f}, R²: {r2:.4f}")

# 4. Which model wins? By how much? Is the difference practically significant?


Best params: {'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 200}

=== Model Comparison ===
Ridge           — RMSE: 0.7455, R²: 0.5759
RF (default)    — RMSE: 0.5053, R²: 0.8051
RF (tuned)      — RMSE: 0.4960, R²: 0.8123
GBR             — RMSE: 0.4736, R²: 0.8288


---

## Extension: SHAP Analysis (5200 depth — 15 min)

Use SHAP to explain individual predictions. Compare MDI ranking vs. SHAP ranking.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: SHAP setup and TreeExplainer
# -----------------------------------------------------------

# Install SHAP if needed
!pip install shap
import shap

# Create SHAP explainer for the tuned RF
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

# pick 3 observations: high-value, low-value, one surprising
preds = best_rf.predict(X_test)
high_idx = np.argmax(preds)    # highest predicted price
low_idx = np.argmin(preds)     # lowest predicted price
mid_idx = np.argsort(preds)[len(preds)//2]  # middle prediction as "surprising"

# 1. Waterfall plot for 3 observations: one high-value, one low-value, one surprising
print("=== High-value observation ===")
shap.plots.waterfall(shap.Explanation(
    values=shap_values[high_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_idx]
))

print("=== Low-value observation ===")
shap.plots.waterfall(shap.Explanation(
    values=shap_values[low_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[low_idx]
))

print("=== Mid/surprising observation ===")
shap.plots.waterfall(shap.Explanation(
    values=shap_values[mid_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[mid_idx]
))

# 2. Beeswarm plot (global view)
shap.plots.beeswarm(shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_test
))

# 3. Compare MDI ranking vs SHAP ranking \u2014 do they agree? Where do they diverge?
shap_imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)

compare = pd.DataFrame({
    'MDI_Rank': importance.rank(ascending=False).astype(int),
    'SHAP_Rank': shap_imp.rank(ascending=False).astype(int)
}).sort_values('SHAP_Rank')
print("\nMDI vs SHAP Ranking:")
print(compare)

**SHAP Interpretation:**

Median income is the strongest predictor of housing prices — higher income areas tend to have higher predicted values. Location also matters a lot: latitude and longitude capture regional price differences, like coastal vs. inland areas. However, these are predictive relationships, not causal ones. We can't say that moving a house to a different location would change its price — the location effect reflects correlated factors like job markets, climate, and amenities that our model doesn't separate.

### SHAP Interpretation (write as a .py module)

Create a reusable `shap_analysis.py` module with:
- `explain_prediction(model, X, idx)` → returns SHAP waterfall for observation `idx`
- `global_importance(model, X)` → returns SHAP beeswarm plot
- `compare_importance(model, X, y)` → returns side-by-side MDI vs SHAP ranking

Include docstrings and type hints. This is a portfolio artifact.

In [13]:
%%writefile shap_utils.py

"""
shap_utils.py — Reusable SHAP explanation functions for tree-based models.
ECON 5200 Lab 19
"""

import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance


def explain_prediction(model, X: pd.DataFrame, idx: int):
    """Generate a SHAP waterfall plot for a single observation.

    Args:
        model: A fitted tree-based model (e.g., RandomForestRegressor).
        X: Feature DataFrame.
        idx: Row index of the observation to explain.

    Returns:
        SHAP waterfall plot for the selected observation.
    """
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X.iloc[[idx]])
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X.iloc[idx]
    ))


def global_importance(model, X: pd.DataFrame):
    """Generate a SHAP beeswarm plot showing global feature importance.

    Args:
        model: A fitted tree-based model.
        X: Feature DataFrame.

    Returns:
        SHAP beeswarm plot.
    """
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap.plots.beeswarm(shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X
    ))


def compare_importance(model, X: pd.DataFrame, y):
    """Compare MDI vs SHAP feature importance rankings side by side.

    Args:
        model: A fitted tree-based model.
        X: Feature DataFrame.
        y: Target values.

    Returns:
        DataFrame with MDI and SHAP rankings.
    """
    # MDI importance
    mdi = pd.Series(model.feature_importances_, index=X.columns)

    # SHAP importance
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap_imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns)

    comparison = pd.DataFrame({
        'MDI': mdi.round(4),
        'MDI_Rank': mdi.rank(ascending=False).astype(int),
        'SHAP': shap_imp.round(4),
        'SHAP_Rank': shap_imp.rank(ascending=False).astype(int)
    }).sort_values('SHAP_Rank')

    print(comparison)
    return comparison


if __name__ == "__main__":
    from sklearn.datasets import fetch_california_housing
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split

    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    print("=== Waterfall for observation 0 ===")
    explain_prediction(rf, X_test.iloc[:50], 0)

    print("=== Global importance ===")
    global_importance(rf, X_test.iloc[:50])

    print("=== MDI vs SHAP ===")
    compare_importance(rf, X_test.iloc[:50], y_test[:50])

Overwriting shap_utils.py


---
## AI-Assisted Expansion: SHAP Dashboard + Reusable Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, feature importance, and SHAP explanations. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/shap_utils.py` module** with:
- `explain_prediction(model, X, idx)` → SHAP waterfall plot
- `global_importance(model, X)` → SHAP beeswarm plot
- `compare_importance(model, X, y)` → side-by-side MDI vs SHAP ranking
- Full docstrings, type hints, and error handling

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See SHAP waterfall + beeswarm plots update with each parameter change
3. Compare RF vs Ridge vs GBR performance as hyperparameters change
4. Toggle between MDI, permutation, and SHAP importance rankings

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [14]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in SHAP explanations, interactive visualizations, and
# scikit-learn production workflows.
#
# [Request] I just completed a diagnosis-first lab where I
# compared Decision Trees, Ridge, Random Forests, and Gradient
# Boosting on California Housing data. I fixed evaluation bugs,
# diagnosed causal overclaiming from MDI, tuned hyperparameters
# with GridSearchCV, and generated SHAP waterfall + beeswarm
# plots. Now I need TWO artifacts:
#
# 1. A reusable `src/shap_utils.py` module with three functions:
#    - explain_prediction(model, X, idx) -> SHAP waterfall
#    - global_importance(model, X) -> SHAP beeswarm
#    - compare_importance(model, X, y) -> MDI vs SHAP side-by-side
#    Include type hints, docstrings, and error handling.
#
# 2. An interactive Plotly dashboard (or Streamlit app) with
#    ipywidgets sliders for n_estimators (1-500) and max_features
#    (1-8). The dashboard should update four panels:
#    (a) model comparison bar chart (RF vs Ridge vs GBR),
#    (b) SHAP beeswarm that updates with max_features,
#    (c) Train vs Test R\u00b2 as n_estimators increases,
#    (d) toggle between MDI / permutation / SHAP rankings.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, shap, numpy,
# sklearn. Use the same variable names: X_train, X_test,
# y_train, y_test, data.feature_names. Do not use deprecated
# Plotly or SHAP functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How TreeExplainer differs from KernelExplainer
#   - Why SHAP values are additive (Shapley property)
#   - How ipywidgets observers trigger plot updates
#   - Why we re-fit inside the callback
#
# [Evaluate] Explain what the dashboard reveals about:
#   - The relationship between n_estimators, max_features,
#     and test performance
#   - Where MDI and SHAP rankings diverge and why
#   - The marginal value of additional trees beyond ~200

# PASTE AI-GENERATED CODE BELOW:


In [15]:
%%writefile streamlit_app.py

"""
Streamlit Dashboard — Lab 19: Random Forest SHAP Explorer
ECON 5200
"""

import streamlit as st
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

st.set_page_config(page_title="Lab 19: RF SHAP Explorer", layout="wide")
st.title("Lab 19: Random Forest SHAP Explorer")

# load data
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sidebar sliders
st.sidebar.header("Hyperparameters")
n_estimators = st.sidebar.slider("n_estimators", 10, 500, 100, step=10)
max_features = st.sidebar.slider("max_features", 1, 8, 4)

# fit models
rf = RandomForestRegressor(n_estimators=n_estimators, max_features=max_features, random_state=42)
rf.fit(X_train, y_train)

ridge = Ridge(alpha=1.0).fit(X_train, y_train)
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)

# model comparison
st.subheader("Model Comparison")
results = {}
for name, model in [("Ridge", ridge), ("Random Forest", rf), ("GBR", gbr)]:
    r2 = r2_score(y_test, model.predict(X_test))
    rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    results[name] = {"R²": round(r2, 4), "RMSE": round(rmse, 4)}
st.dataframe(pd.DataFrame(results).T)

# SHAP analysis on small sample
st.subheader("SHAP Analysis")
X_sample = X_test.iloc[:50]
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)

# beeswarm
st.write("**Beeswarm Plot (Global Feature Importance)**")
fig_bee, ax_bee = plt.subplots()
shap.plots.beeswarm(shap.Explanation(
    values=shap_values, base_values=explainer.expected_value, data=X_sample
), show=False)
st.pyplot(fig_bee)

# waterfall for selected observation
obs_idx = st.slider("Select observation for waterfall plot", 0, 49, 0)
st.write(f"**Waterfall Plot — Observation {obs_idx}**")
fig_wf, ax_wf = plt.subplots()
shap.plots.waterfall(shap.Explanation(
    values=shap_values[obs_idx],
    base_values=explainer.expected_value,
    data=X_sample.iloc[obs_idx]
), show=False)
st.pyplot(fig_wf)

# importance ranking toggle
st.subheader("Feature Importance Ranking")
method = st.radio("Select method:", ["MDI", "SHAP"])
if method == "MDI":
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
else:
    imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)
st.bar_chart(imp)

Overwriting streamlit_app.py


In [16]:
%%writefile verification-log.md

# P.R.I.M.E. Verification Log — Lab 19

## Prompt
Used the P.R.I.M.E. prompt from the lab instructions to generate:
1. `src/shap_utils.py` — reusable SHAP module with three functions
2. `streamlit_app.py` — interactive dashboard with sliders and SHAP plots

## What AI Generated
- shap_utils.py with explain_prediction(), global_importance(), compare_importance()
- Streamlit app with n_estimators/max_features sliders, model comparison table, beeswarm plot, waterfall plot, and MDI/SHAP toggle

## What I Changed
- Reduced SHAP sample size to 50 observations for speed
- Verified function signatures match lab requirements
- Confirmed all three functions have docstrings and type hints

## What I Verified
- shap_utils.py creates successfully with %%writefile
- Functions follow the required API: explain_prediction(model, X, idx), global_importance(model, X), compare_importance(model, X, y)
- MDI and SHAP rankings both show MedInc as top predictor
- Interpretation is non-causal (prediction only, not policy recommendation)

Overwriting verification-log.md


---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [17]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

In [18]:
%%writefile README.md

# Lab 19: Tree-Based Models — Random Forests

## Objective
Compare tree-based models against linear regression on California Housing data, diagnose common evaluation and interpretation errors, and explain predictions using SHAP.

## Methodology
- Loaded California Housing dataset (20,640 observations, 8 features)
- Compared Decision Tree, Ridge Regression, Random Forest, and Gradient Boosting
- Diagnosed a train/test evaluation bug that inflated RF performance
- Critiqued a causal overclaim from MDI feature importance
- Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth, max_features)
- Generated SHAP waterfall and beeswarm plots for model interpretability
- Built a reusable shap_utils.py module and interactive Streamlit dashboard

## Key Findings
- Random Forest Test R² = 0.8051, Gradient Boosting R² = 0.8288, Ridge R² = 0.5759
- GBR slightly outperforms tuned RF; both substantially beat Ridge
- MedInc is the top predictor in both MDI and SHAP rankings
- Feature importance captures predictive power, not causal effects

## How to Reproduce
```
pip install -r requirements.txt
jupyter notebook notebooks/lab_19_random_forests.ipynb
```

## Repository Structure
```
econ-lab-19-random-forests/
├── README.md
├── requirements.txt
├── notebooks/
│   └── lab_19_random_forests.ipynb
├── src/
│   └── shap_utils.py
├── figures/
│   ├── shap_waterfall.png
│   ├── shap_beeswarm.png
│   └── feature_importance.png
└── verification-log.md
```

Overwriting README.md


In [19]:
%%writefile requirements.txt

numpy
pandas
scikit-learn
matplotlib
shap
streamlit

Overwriting requirements.txt


### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.